In [29]:
# Basic packages
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd
import math
from netCDF4 import Dataset, num2date

# DateTime packages
from matplotlib.dates import DateFormatter
from datetime import datetime, timedelta
import time
import matplotlib.dates as mdates

# Stats packages
import scipy
import PyCO2SYS as pyco2
import gsw
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
import optuna

# Logistical packages
import requests
from importlib import reload
import warnings
import re
from pathlib import Path
import gc

# 1. Configuration

This biological calibration notebook excludes temperature and salinity from both the diagnostic metrics and total cost. It adds two station-file predictors to the existing biological parameter suite: `nl_tnu2_tracer9` and the paired horizontal/vertical advection scheme reported for `detritus` in `NLM_TADV`.

Tracer 9 is used as a representative of all biological tracers. When candidate files are written, its suggested `nl_tnu2` and advection pair are applied uniformly to all 15 biological tracers. Database updates, new candidate generation, file writing, and run-summary export all default to off.

In [30]:
# ============================================================
# PATHS
# ============================================================

COST_DIR = Path("/Users/akbaskind/Desktop/COST_FILES")
OPT_DIR = Path("/Users/akbaskind/Desktop/Optimization")
FILE_LIST = OPT_DIR / "FileNames.xlsx"
PARAMETER_LIST = OPT_DIR / "ParameterList copy.csv"
RUN_MAP_FILE = OPT_DIR / "runmap.xlsx"
DSTART_VARIABLE = "dstart"
DSTART_YEAR_COLUMN = "dstart_year"
HISTORY_ATTRIBUTE = "history"
RUN_DATE_COLUMN = "Run Date"
RUN_DATE_FALLBACK = pd.Timestamp("2000-01-01")

runs = pd.read_excel(FILE_LIST)
runs = runs[["Run Name", "Cost File", "Station File"]].dropna(subset=["Run Name", "Cost File", "Station File"])

parameter_names = (
    pd.read_csv(PARAMETER_LIST)["Parameter Name"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

print(f"{len(runs)} runs")
print(f"{len(parameter_names)} parameters")

# ============================================================
# TARGET SELECTION
# ============================================================

include_benthic = True
exclude_some_benthic = True

benthic_cols = ["SOD",
                "Benthic NO3 Flux",
                "Benthic NH4 Flux",
               ]

some_benthic_cols = ["Benthic NO3 Flux",
                     "Benthic NH4 Flux",
                    ]

# ============================================================
# FILTERS AND THRESHOLDS
# ============================================================

COST_THRESHOLD = 200

# ============================================================
# METRIC DEFINITIONS FOR nRMSE
# ============================================================

metric_info = {
    "Surface pH": ("pH_mod", "pH_obs"),
    "Bottom pH":  ("pH_mod", "pH_obs"),
    # "Total pH":   ("pH_mod", "pH_obs"),

    "Surface Oxygen": ("oxygen_mod", "oxygen_obs"),
    "Bottom Oxygen":  ("oxygen_mod", "oxygen_obs"),
    # "Total Oxygen":   ("oxygen_mod", "oxygen_obs"),

    "Surface NO3": ("NO3_mod", "NO3_obs"),
    "Bottom NO3":  ("NO3_mod", "NO3_obs"),
    # "Total NO3":   ("NO3_mod", "NO3_obs"),

    "Surface NH4": ("NH4_mod", "NH4_obs"),
    "Bottom NH4":  ("NH4_mod", "NH4_obs"),
    # "Total NH4":   ("NH4_mod", "NH4_obs"),

    "Surface Si": ("Si_mod", "Si_obs"),
    "Bottom Si":  ("Si_mod", "Si_obs"),
    # "Total Si":   ("Si_mod", "Si_obs"),

    "Secchi Depth": ("SD_mod", "SD_obs"),

    "SOD": ("sed_o2_mod", "sed_o2_obs"),
    "Benthic NO3 Flux": ("sed_no3_mod", "sed_no3_obs"),
    "Benthic NH4 Flux": ("sed_nh4_mod", "sed_nh4_obs"),
}

if not include_benthic:
    metric_info = dict([i for i in metric_info.items() if i[0] not in benthic_cols])
elif exclude_some_benthic:
    metric_info = dict([i for i in metric_info.items() if i[0] not in some_benthic_cols])
    

print(f"\nNumber of nRMSE metrics = {len(metric_info)}")
print(f"nRMSE metrics: {list(metric_info.keys())}")

# ============================================================
# METRIC DEFINITIONS FOR COST
# ============================================================

cost_components = {"Surface pH": "pH_cost",
                   "Bottom pH": "pH_cost", 
                   "Surface Oxygen": "oxygen_cost",
                   "Bottom Oxygen": "oxygen_cost",
                   "Surface NO3": "NO3_cost",
                   "Bottom NO3": "NO3_cost",
                   "Surface NH4": "NH4_cost",
                   "Bottom NH4": "NH4_cost",
                   "Surface Si": "Si_cost",
                   "Bottom Si": "Si_cost",
                   "Secchi Depth": "SD_cost",
                   "SOD": "sed_o2_cost",
                   "Benthic NO3 Flux": "sed_no3_cost",
                   "Benthic NH4 Flux": "sed_nh4_cost"
                  }

cost_weights    = {"Surface pH": 2,
                   "Bottom pH": 2, 
                   "Surface Oxygen": 2,
                   "Bottom Oxygen": 2,
                   "Surface NO3": 1,
                   "Bottom NO3": 1,
                   "Surface NH4": 1,
                   "Bottom NH4": 1,
                   "Surface Si": 1,
                   "Bottom Si": 1,
                   "Secchi Depth": 1,
                   "SOD": 1,
                   "Benthic NO3 Flux": 1,
                   "Benthic NH4 Flux": 1
                  }

if not include_benthic:
    cost_components = dict([i for i in cost_components.items() if i[0] not in benthic_cols])
    cost_weights = dict([i for i in cost_weights.items() if i[0] not in benthic_cols])
elif exclude_some_benthic:
    cost_components = dict([i for i in cost_components.items() if i[0] not in some_benthic_cols])
    cost_weights = dict([i for i in cost_weights.items() if i[0] not in some_benthic_cols])

print(f"\nNumber of cost targets = {len(cost_components)}")
print(f"Cost metrics: {list(cost_components.keys())}")

# ============================================================
# CANDIDATE BASELINE SETTINGS
# ============================================================

BASELINE_RUN_NAME = "Dave5_2005"

# These values take precedence over the Dave5 baseline
# but do not modify the original template files.
SCIENTIFIC_OVERRIDES = {
    "bao2": 2.0,
}

# ============================================================
# REPRESENTATIVE BIOLOGICAL TRACER SETTINGS
# ============================================================

BIO_TRACER_INDEX = 9
BIO_TRACER_ATTRIBUTE_NAME = "detritus"
BIO_NL_TNU2_PARAMETER = "nl_tnu2_tracer9"
BIO_ADVECTION_PARAMETER = "tracer9_advection_scheme"
BIOLOGICAL_TRACER_COUNT = 15

# Categories use descriptive station-file names. Input-file aliases are
# applied only when candidate text is generated.
BIO_ADVECTION_SCHEMES = {
    "Upstream3_Centered4": ("Upstream3", "Centered4"),
    "HSIMT": ("HSIMT", "HSIMT"),
    "Akima4": ("Akima4", "Akima4"),
    "MPDATA": ("MPDATA", "MPDATA"),
}
ADVECTION_INPUT_CODES = {
    "Upstream3": "U3",
    "Centered4": "C4",
    "Akima4": "A4",
    "MPDATA": "MPDATA",
    "HSIMT": "HSIMT",
}

# ============================================================
# OPTUNA STUDY SETTINGS
# ============================================================

# Separate study because this cost excludes temperature and salinity and
# includes representative biological mixing/advection parameters.
STUDY_NAME = "model_calibration_bio"
CURRENT_STUDY_NAME = STUDY_NAME

STORAGE_FILE = OPT_DIR / "model_calibration_bio.db"

STORAGE = f"sqlite:///{STORAGE_FILE}"

# Load and validate the shared mapping from study-assigned names to model run names.
required_run_map_columns = [
    "Actual Run Name", "Study Name", "Study Run Name", "Trial Number"
]
run_map = pd.read_excel(RUN_MAP_FILE)
run_map.columns = run_map.columns.astype(str).str.strip()
missing_run_map_columns = [
    column for column in required_run_map_columns if column not in run_map.columns
]
if missing_run_map_columns:
    raise ValueError(
        f"Run map is missing required columns: {missing_run_map_columns}"
    )

for column in ["Actual Run Name", "Study Name", "Study Run Name"]:
    run_map[column] = run_map[column].astype("string").str.strip()

trial_numbers = pd.to_numeric(run_map["Trial Number"], errors="coerce")
invalid_trial_numbers = trial_numbers.isna() | (trial_numbers % 1 != 0)
if invalid_trial_numbers.any():
    invalid_rows = (run_map.index[invalid_trial_numbers] + 2).tolist()
    raise ValueError(
        f"Run map has invalid Trial Number values in Excel rows: {invalid_rows}"
    )
run_map["Trial Number"] = trial_numbers.astype(int)

current_study_run_map = run_map.loc[
    run_map["Study Name"].eq(CURRENT_STUDY_NAME)
].copy()
duplicate_mapping_keys = [
    ["Study Name", "Trial Number"],
    ["Study Name", "Study Run Name"],
    ["Study Name", "Actual Run Name"],
]
duplicate_mapping_messages = []
for columns in duplicate_mapping_keys:
    duplicate_rows = current_study_run_map.loc[
        current_study_run_map.duplicated(columns, keep=False), columns
    ]
    if not duplicate_rows.empty:
        duplicate_mapping_messages.append(
            f"{columns}: {duplicate_rows.to_dict('records')}"
        )
if duplicate_mapping_messages:
    raise ValueError(
        f"Invalid duplicate run-map mappings for {CURRENT_STUDY_NAME!r}:\n"
        + "\n".join(duplicate_mapping_messages)
    )

print(f"\nStudy name: {STUDY_NAME}")
print(f"Storage:    {STORAGE_FILE}")
print(f"Run-map rows for this study: {len(current_study_run_map)}")

# Set True to tell Optuna about RUNNING trials whose model results now exist.
# When False, eligible trials are reported but the study is not modified.
UPDATE_OPTUNA_WITH_COMPLETED_RUNS = False

# Set True only when a new batch of Optuna suggestions is needed.
GENERATE_NEW_CANDIDATES = False
N_CANDIDATES = 4

# This is independent of candidate generation, allowing files for existing
# RUNNING candidates to be regenerated without asking Optuna for new trials.
WRITE_CANDIDATE_INPUT_FILES = False

# Export the cleaned model-run table used by downstream analysis.
EXPORT_RUN_SUMMARY = False

# Maximum number of mismatched model/observation locations printed per metric.
MAX_MISMATCH_LOCATIONS = 10

# Identifies the exact cost-function definition used for this study
OBJECTIVE_VERSION = "cost_v3_bio_no_temp_salt"

110 runs
74 parameters

Number of nRMSE metrics = 12
nRMSE metrics: ['Surface pH', 'Bottom pH', 'Surface Oxygen', 'Bottom Oxygen', 'Surface NO3', 'Bottom NO3', 'Surface NH4', 'Bottom NH4', 'Surface Si', 'Bottom Si', 'Secchi Depth', 'SOD']

Number of cost targets = 12
Cost metrics: ['Surface pH', 'Bottom pH', 'Surface Oxygen', 'Bottom Oxygen', 'Surface NO3', 'Bottom NO3', 'Surface NH4', 'Bottom NH4', 'Surface Si', 'Bottom Si', 'Secchi Depth', 'SOD']

Study name: model_calibration_bio
Storage:    /Users/akbaskind/Desktop/Optimization/model_calibration_bio.db
Run-map rows for this study: 0


# 2. Functions

## 2.1. nRMSE functions

$$
RMSD^{*'} = \text{sign}(\Delta \sigma)\sqrt{1 + {(\frac{\sigma_a}{\sigma_{\widehat{a}}})}^2 -2(\frac{\sigma_a}{\sigma_{\widehat{a}}}) \frac{\frac{1}{N} \sum_{n=1}^{N} (a_n - \overline{a})(\widehat{a_n} - \overline{\widehat{a}})}{\sigma_a \sigma_{\widehat{a}}}}
$$

* $\widehat{a}$: observations
* $a$: model results

From [Jolliff et al. (2009)](https://doi.org/10.1016/j.jmarsys.2008.05.014).

Model and observation values are expected to share dimensions and missing-data locations. Each metric is calculated from paired finite values only. Any mismatch is printed with counts and coordinate locations (up to `MAX_MISMATCH_LOCATIONS`) and retained in a diagnostics table.

In [31]:
# ============================================================
# OBSERVATIONAL ERROR
# ============================================================

def get_stderr(var_obs, obs_mean):

    if var_obs.startswith("sed"):
        return obs_mean

    elif var_obs.startswith("pH"):
        return 0.1

    elif var_obs.startswith("temp"):
        return 0.01

    elif var_obs.startswith("salt"):
        return 0.005 * obs_mean

    elif var_obs.startswith("oxy"):
        return 3.125

    elif var_obs.startswith("N"):
        return 0.1

    elif var_obs.startswith("Si"):
        return 0.1

    elif var_obs.startswith("SD"):
        return 0.2

    else:
        return 0


# ============================================================
# CALCULATE ONE RMSE*
# ============================================================

def describe_locations(data_array, mask, limit):
    """Return readable coordinate labels for True entries in a mask."""

    locations = []

    for index_values in np.argwhere(mask)[:limit]:
        labels = []

        for dim, index in zip(data_array.dims, index_values):
            if dim in data_array.coords and data_array.coords[dim].ndim == 1:
                coordinate = data_array.coords[dim].values[index]
                labels.append(f"{dim}={coordinate}")
            else:
                labels.append(f"{dim}[{index}]")

        locations.append(", ".join(labels) if labels else "scalar")

    return locations


def record_rmse_issue(
    diagnostics,
    run_name,
    metric_name,
    issue,
    details,
    print_details=True,
):
    """Print and retain one RMSE data-quality issue."""

    record = {
        "Run Name": run_name,
        "Metric": metric_name,
        "Issue": issue,
        "Details": details,
    }
    diagnostics.append(record)
    if print_details:
        print(f"  RMSE WARNING [{metric_name}] {issue}: {details}")


def calculate_rmse_star(
    ds,
    metric_name,
    var_mod,
    var_obs,
    run_name,
    diagnostics,
    max_mismatch_locations=10,
):
    """Calculate signed RMSD*' from paired, finite values."""

    missing_variables = [
        variable
        for variable in (var_mod, var_obs)
        if variable not in ds.variables
    ]

    if missing_variables:
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Missing variable",
            f"not found: {missing_variables}",
        )
        return np.nan

    mod_data = ds[var_mod]
    obs_data = ds[var_obs]

    if metric_name.startswith("Surface"):
        mod_data = mod_data.isel(Depth=0)
        obs_data = obs_data.isel(Depth=0)
    elif metric_name.startswith("Bottom"):
        mod_data = mod_data.isel(Depth=1)
        obs_data = obs_data.isel(Depth=1)

    # Pairing is meaningful only when dimensions and shapes agree exactly.
    if mod_data.dims != obs_data.dims or mod_data.shape != obs_data.shape:
        details = (
            f"model dims/shape={mod_data.dims}/{mod_data.shape}; "
            f"observation dims/shape={obs_data.dims}/{obs_data.shape}"
        )
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Alignment mismatch", details
        )
        return np.nan

    mod_values = np.asarray(mod_data.values)
    obs_values = np.asarray(obs_data.values)
    mod_finite = np.isfinite(mod_values)
    obs_finite = np.isfinite(obs_values)

    model_only = mod_finite & ~obs_finite
    observation_only = ~mod_finite & obs_finite

    if model_only.any() or observation_only.any():
        model_locations = describe_locations(
            mod_data, model_only, max_mismatch_locations
        )
        observation_locations = describe_locations(
            obs_data, observation_only, max_mismatch_locations
        )
        details = (
            f"model finite/observation non-finite={model_only.sum()} "
            f"at {model_locations}; "
            f"observation finite/model non-finite={observation_only.sum()} "
            f"at {observation_locations}"
        )
        record_rmse_issue(
            diagnostics,
            run_name,
            metric_name,
            "Missing-data mismatch",
            details,
            print_details=False,
        )
        print(
            f"  RMSE WARNING [{metric_name}] missing-data mismatch: "
            f"model-only={model_only.sum()}, "
            f"observation-only={observation_only.sum()}"
        )

    # Use the same paired sample for every statistic.
    paired = mod_finite & obs_finite
    mod = mod_values[paired].astype(float)
    obs = obs_values[paired].astype(float)

    if mod.size < 2:
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Insufficient paired data",
            f"found {mod.size} paired finite value(s); at least 2 are required",
        )
        return np.nan

    mod_mean = mod.mean()
    obs_mean = obs.mean()
    std_mod = mod.std()
    std_obs_sample = obs.std()

    # Include the configured observational uncertainty in observation spread.
    stderr = get_stderr(var_obs, obs_mean)
    std_obs = np.sqrt(std_obs_sample**2 + stderr**2)

    if std_mod == 0 or not np.isfinite(std_mod):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid model variance",
            f"model standard deviation={std_mod}",
        )
        return np.nan

    if std_obs == 0 or not np.isfinite(std_obs):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid observation variance",
            f"adjusted observation standard deviation={std_obs}",
        )
        return np.nan

    covariance = np.mean((mod - mod_mean) * (obs - obs_mean))
    correlation = covariance / (std_mod * std_obs)
    std_norm = std_mod / std_obs
    sign = np.sign(std_mod - std_obs)

    inside = 1 + std_norm**2 - 2 * std_norm * correlation

    # Roundoff can produce a tiny negative value; a larger one is not valid.
    if inside < -1e-12 or not np.isfinite(inside):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid RMSD expression",
            f"value under square root={inside}",
        )
        return np.nan

    rmsd_star = np.sqrt(max(inside, 0.0)) * sign
    return float(rmsd_star)

## 2.2. Cost functions

The cost function is based on that found in Equation 1 of [Ward et al. (2010)](https://www.sciencedirect.com/science/article/pii/S0924796309003431?casa_token=S1tQ2yuwGQgAAAAA:lCpjjVEqP4bJWA4xg5kOjG028PuyDQdcOHfTw5KtVLoM0kkonEa15ZinoxIopsWH1nTmhNRExw).

$$
J = \frac{1}{M} \sum_{m=1}^{M} W_m^2 \frac{1}{N_m} \sum_{n=1}^{N_m} (a - \hat{a})_{n,m}^2
$$

$M$: number of data targets \
$N_m$: number of observations for each target \
$\hat{a}$: observed value of data target $m$ at location/time $n$ \
$a$: model equivalent of $\hat{a}$ \
$W_m$: weight function; $W_m = \frac{C_m}{\sigma_m}$

**Implementation notes:** The $\frac{1}{M}$ term is omitted, so comparisons assume every simulation uses the same targets. Depth-resolved cost variables use index 0 for the surface target and index 1 for the bottom target.

In [32]:
def calculate_cost(ds, cost_components, cost_weights):
    """Sum weighted surface, bottom, and non-depth-specific costs."""

    total_cost = 0.0

    for target, variable_name in cost_components.items():
        weight = cost_weights[target]

        if target.startswith("Surface"):
            component = float(ds[variable_name][0])
        elif target.startswith("Bottom"):
            component = float(ds[variable_name][1])
        else:
            component = float(ds[variable_name])

        total_cost += component * weight**2

    return total_cost

## 2.3. Parameter extraction functions

Method to extract parameter values from the station file. The `dstart` time variable is decoded separately and retained as the numeric parameter `dstart_year`.

In [33]:
def get_parameter_values(ds, parameter_name):

    # Parameter doesn't exist in this file
    # Don't create a new scalar-named column
    if parameter_name not in ds.variables:
        return {}

    var = ds.variables[parameter_name]

    try:

        # --------------------------------
        # Scalar parameter
        # --------------------------------
        if var.ndim == 0:

            value = var[...]

            if np.ma.is_masked(value):
                value = np.nan

            return {
                parameter_name: np.asarray(value).item()
            }

        # --------------------------------
        # Parameter with dimensions
        # --------------------------------
        values = np.asarray(var[:]).squeeze()

        # Squeezes down to a scalar
        if values.ndim == 0:

            return {
                parameter_name: values.item()
            }

        # --------------------------------
        # One-dimensional parameter
        # --------------------------------
        if values.ndim == 1:

            dim_name = var.dimensions[0]

            return {
                f"{parameter_name}_{dim_name}{i}": value
                for i, value in enumerate(values)
            }

        # --------------------------------
        # Anything else
        # --------------------------------
        values = values.ravel()

        return {
            f"{parameter_name}_{i}": value
            for i, value in enumerate(values)
        }

    except Exception as e:

        print(
            f"    Could not read {parameter_name}: {e}"
        )

        return {}


def get_dstart_year(ds, variable_name=DSTART_VARIABLE):
    """Decode the station-file start date and return only its year."""

    if variable_name not in ds.variables:
        return np.nan

    var = ds.variables[variable_name]
    raw_value = var[...]

    if np.ma.is_masked(raw_value):
        return np.nan

    value = np.asarray(raw_value).squeeze()
    if value.size != 1:
        return np.nan

    if np.issubdtype(value.dtype, np.datetime64):
        timestamp = pd.to_datetime(value.reshape(1))[0]
    else:
        units = getattr(var, "units", None)
        if units is None:
            raise ValueError(f"{variable_name} has no NetCDF time units")
        timestamp = num2date(
            value.item(),
            units=units,
            calendar=getattr(var, "calendar", "standard"),
            only_use_cftime_datetimes=False,
        )

    return int(timestamp.year)


def get_run_date(ds, attribute_name=HISTORY_ATTRIBUTE):
    """Extract the model run date from the station-file history attribute."""

    # ROMS history strings contain a date such as 'January 18, 2026'.
    # Return the documented fallback date when the attribute is absent or malformed.
    history = getattr(ds, attribute_name, None)
    if history is None:
        return RUN_DATE_FALLBACK, True

    date_match = re.search(
        r"\b(?:January|February|March|April|May|June|July|August|"
        r"September|October|November|December)\s+\d{1,2},\s+\d{4}\b",
        str(history),
    )
    if date_match is None:
        return RUN_DATE_FALLBACK, True

    run_date = pd.to_datetime(
        date_match.group(0),
        format="%B %d, %Y",
        errors="coerce",
    )
    if pd.isna(run_date):
        return RUN_DATE_FALLBACK, True

    return run_date.normalize(), False

## 2.4. Representative biological-tracer extraction

`nl_tnu2_tracer9` is read from the numeric `nl_tnu2` station-file variable. The paired categorical advection scheme is read from the `detritus:` line of the multiline `NLM_TADV` global attribute. Complete but unconfigured pairs are retained as missing and reported before trial import.

In [34]:
def get_biological_advection_scheme(
    ds,
    tracer_name=BIO_TRACER_ATTRIBUTE_NAME,
    attribute_name="NLM_TADV",
):
    """Return the configured category matching one tracer's H/V pair."""

    if attribute_name not in ds.ncattrs():
        return np.nan

    match = re.search(
        rf"^\s*{re.escape(tracer_name)}:\s+(\S+)\s+(\S+)",
        str(ds.getncattr(attribute_name)),
        flags=re.MULTILINE,
    )
    if match is None:
        return np.nan

    observed_pair = match.groups()
    for category, expected_pair in BIO_ADVECTION_SCHEMES.items():
        if observed_pair == expected_pair:
            return category

    return np.nan

# 3. Get nRMSE, cost, and parameter values

Cycles through all the runs listed in `FILE_LIST`, opens the cost summary file to calculate nRMSE and cost, and opens the station file to get the parameters and decoded `dstart_year`. Any runs in `FILE_LIST` that are missing a file or a metric or a parameter will be filtered out. 

In [35]:
# ============================================================
# PROCESS EACH MODEL RUN ONCE
#   - RMSE metrics
#   - Cost
#   - Parameters
# ============================================================

all_results = []
rmse_diagnostics = []

missing_cost_files = []
missing_station_files = []
fallback_run_date_runs = []

for n, (_, row) in enumerate(runs.iterrows(), start=1):

    run_name = row["Run Name"]
    cost_file = row["Cost File"]
    station_file = row["Station File"]

    cost_path = COST_DIR / cost_file
    station_path = COST_DIR / station_file

    print(f"\n[{n}/{len(runs)}] Processing {run_name}")
    print(f"  Cost file:    {cost_path.name}")
    print(f"  Station file: {station_path.name}")

    # Everything for this run goes into ONE dictionary
    run_results = {
        "Run Name": run_name,
        "Cost File": cost_file,
        "Station File": station_file
    }

    # ========================================================
    # COST FILE
    #   1. Calculate RMSE metrics
    #   2. Calculate total cost
    # ========================================================

    try:

        with xr.open_dataset(cost_path) as ds:

            # --------------------------
            # RMSE metrics
            # --------------------------
            for metric_name, (var_mod, var_obs) in metric_info.items():

                rmse = calculate_rmse_star(
                    ds,
                    metric_name,
                    var_mod,
                    var_obs,
                    run_name,
                    rmse_diagnostics,
                    MAX_MISMATCH_LOCATIONS,
                )

                run_results[metric_name] = rmse

            # --------------------------
            # Combined cost
            # --------------------------
            cost = calculate_cost(
                ds,
                cost_components,
                cost_weights
            )

            run_results["Cost"] = cost

        run_results["Cost Status"] = "Success"

    except Exception as e:

        print(f"  COST FILE ERROR: {e}")

        run_results["Cost Status"] = f"ERROR: {e}"

        # Make sure expected outputs exist
        for metric_name in metric_info:
            run_results.setdefault(metric_name, np.nan)

        run_results.setdefault("Cost", np.nan)

        missing_cost_files.append(cost_path.name)

    # ========================================================
    # STATION FILE
    #   Extract model parameters
    # ========================================================

    try:

        with Dataset(station_path, mode="r") as ds:

            for parameter in parameter_names:

                parameter_values = get_parameter_values(
                    ds,
                    parameter
                )

                run_results.update(parameter_values)

            run_results[DSTART_YEAR_COLUMN] = get_dstart_year(ds)

            # Preserve the run date recorded in this station file's history.
            run_date, used_run_date_fallback = get_run_date(ds)
            run_results[RUN_DATE_COLUMN] = run_date
            if used_run_date_fallback:
                fallback_run_date_runs.append(run_name)

            # nl_tnu2 is not part of the biological parameter list, so read
            # tracer9 explicitly from the station-file variable.
            nl_tnu2_values = get_parameter_values(ds, "nl_tnu2")
            run_results[BIO_NL_TNU2_PARAMETER] = nl_tnu2_values.get(
                BIO_NL_TNU2_PARAMETER,
                np.nan,
            )

            # Use detritus (station-file tracer9) as the representative
            # horizontal/vertical biological advection pair.
            run_results[BIO_ADVECTION_PARAMETER] = (
                get_biological_advection_scheme(ds)
            )

        run_results["Parameter Status"] = "Success"

    except Exception as e:

        print(f"  STATION FILE ERROR: {e}")

        run_results["Parameter Status"] = f"ERROR: {e}"

        # Fill parameters with NaN if station file cannot be read
        for parameter in parameter_names:
            run_results.setdefault(parameter, np.nan)

        run_results.setdefault(DSTART_YEAR_COLUMN, np.nan)
        run_results.setdefault(RUN_DATE_COLUMN, RUN_DATE_FALLBACK)
        if run_name not in fallback_run_date_runs:
            fallback_run_date_runs.append(run_name)
        run_results.setdefault(BIO_NL_TNU2_PARAMETER, np.nan)
        run_results.setdefault(BIO_ADVECTION_PARAMETER, np.nan)

        missing_station_files.append(station_path.name)

    # ========================================================
    # SAVE THIS RUN
    # ========================================================

    all_results.append(run_results)

    gc.collect()


# ============================================================
# CREATE FINAL DATAFRAME
# ============================================================

DF_opt = pd.DataFrame(all_results)

# Safety cleanup
DF_opt = DF_opt.dropna(
    axis=1,
    how="all"
)

if fallback_run_date_runs:
    print(
        f"Runs assigned fallback date {RUN_DATE_FALLBACK.date()}: "
        f"{fallback_run_date_runs}"
    )


[1/110] Processing A13
  Cost file:    A13.nc
  Station file: ocean_sta_A13.nc
  RMSE WARNING [Surface pH] missing-data mismatch: model-only=1064, observation-only=0
  RMSE WARNING [Bottom pH] missing-data mismatch: model-only=1472, observation-only=0
  RMSE WARNING [Surface Oxygen] missing-data mismatch: model-only=1064, observation-only=0
  RMSE WARNING [Bottom Oxygen] missing-data mismatch: model-only=1472, observation-only=0
  RMSE WARNING [Bottom Si] missing-data mismatch: model-only=6, observation-only=0
  RMSE WARNING [SOD] missing-data mismatch: model-only=31, observation-only=0

[2/110] Processing DU_Mar16
  Cost file:    DU_Mar16.nc
  Station file: ocean_sta_DU_Mar16.nc
  RMSE WARNING [Surface pH] missing-data mismatch: model-only=28, observation-only=0
  RMSE WARNING [Bottom pH] missing-data mismatch: model-only=42, observation-only=0
  RMSE WARNING [Surface Oxygen] missing-data mismatch: model-only=28, observation-only=0
  RMSE WARNING [Bottom Oxygen] missing-data mismatch

## 3.1. Data-loading and RMSE diagnostics

This section reports failed input files, unusually large costs, missing metric values, and any model/observation pairing problems found while calculating RMSD*'.

In [36]:
failed_cost_runs = DF_opt.loc[
    DF_opt["Cost Status"] != "Success", "Run Name"
].tolist()
failed_parameter_runs = DF_opt.loc[
    DF_opt["Parameter Status"] != "Success", "Run Name"
].tolist()
high_cost_runs = DF_opt.loc[
    DF_opt["Cost"] >= COST_THRESHOLD, "Run Name"
].tolist()

print(f"Runs with cost-file errors: {failed_cost_runs}")
print(f"Runs with station-file errors: {failed_parameter_runs}")
print(f"Runs at or above the cost threshold: {high_cost_runs}")

DF_rmse_diagnostics = pd.DataFrame(
    rmse_diagnostics,
    columns=["Run Name", "Metric", "Issue", "Details"],
)

if DF_rmse_diagnostics.empty:
    print("\nNo RMSE pairing or variance problems found.")
else:
    print(f"\nRMSE diagnostic records: {len(DF_rmse_diagnostics)}")
    display(DF_rmse_diagnostics)

Runs with cost-file errors: ['2005_CTRL', 'C4', 'D16', 'A3', 'A2', 'D17', 'C13', 'A9', 'A14']
Runs with station-file errors: ['C13']
Runs at or above the cost threshold: ['A13', 'A12', 'D10', 'A1', 'D14', 'D4', 'A5', 'D5', 'A4', 'D11', 'A7', 'D6', 'D12', 'D13', 'D10-2', 'A6', 'D7', 'A11', 'D18', 'D8', 'A10', 'D19', 'D9', 'A8']

RMSE diagnostic records: 639


,Run Name,Metric,Issue,Details
0,A13,Surface pH,Missing-data mismatch,model finite/observation non-finite=1064 at ['...
1,A13,Bottom pH,Missing-data mismatch,model finite/observation non-finite=1472 at ['...
2,A13,Surface Oxygen,Missing-data mismatch,model finite/observation non-finite=1064 at ['...
3,A13,Bottom Oxygen,Missing-data mismatch,model finite/observation non-finite=1472 at ['...
4,A13,Bottom Si,Missing-data mismatch,model finite/observation non-finite=6 at ['Dat...
...,...,...,...,...
634,OPTUNA_67,Bottom pH,Missing-data mismatch,model finite/observation non-finite=1472 at ['...
635,OPTUNA_67,Surface Oxygen,Missing-data mismatch,model finite/observation non-finite=1064 at ['...
636,OPTUNA_67,Bottom Oxygen,Missing-data mismatch,model finite/observation non-finite=1472 at ['...
637,OPTUNA_67,Bottom Si,Missing-data mismatch,model finite/observation non-finite=6 at ['Dat...


In [37]:
print("Missing values per RMSE column:")
rmse_columns = list(metric_info)
print(DF_opt[rmse_columns].isna().sum())

print("\nAll values finite:")
print(np.isfinite(DF_opt[rmse_columns]).all())

Missing values per RMSE column:
Surface pH        4
Bottom pH         4
Surface Oxygen    4
Bottom Oxygen     4
Surface NO3       4
Bottom NO3        4
Surface NH4       4
Bottom NH4        4
Surface Si        5
Bottom Si         5
Secchi Depth      4
SOD               9
dtype: int64

All values finite:
Surface pH        False
Bottom pH         False
Surface Oxygen    False
Bottom Oxygen     False
Surface NO3       False
Bottom NO3        False
Surface NH4       False
Bottom NH4        False
Surface Si        False
Bottom Si         False
Secchi Depth      False
SOD               False
dtype: bool


# 4. Clean dataframe for objective nRMSEs and varied parameters

## 4.1. Objective nRMSE 
Absolute value of signed nRMSE so Optuna can minimize the nRMSE

In [38]:
active_rmse_cols = list(metric_info)

active_objective_cols = []

for col in active_rmse_cols:
    new_col = f"{col} Objective"
    DF_opt[new_col] = DF_opt[col].abs()
    active_objective_cols.append(new_col)
    
DF_opt[active_objective_cols].describe().T[["min",
                                            "25%",
                                            "50%",
                                            "75%",
                                            "max"]]

,min,25%,50%,75%,max
Surface pH Objective,0.877419,0.949110,1.006822,1.326534,5.838464
Bottom pH Objective,0.777042,0.893612,1.131900,1.511724,6.676514
Surface Oxygen Objective,0.427493,0.517666,0.593455,0.741815,0.925984
Bottom Oxygen Objective,0.423847,0.576017,0.645140,0.772974,0.995101
Surface NO3 Objective,0.597800,0.796884,0.924099,1.392526,3.266269
Bottom NO3 Objective,0.629221,0.847721,1.042430,1.475975,3.022346
Surface NH4 Objective,0.778513,0.872060,0.912786,0.995499,2.965917
Bottom NH4 Objective,0.940208,1.033298,1.051880,1.256157,3.422291
Surface Si Objective,0.687857,0.805261,0.882835,0.908052,1.030878
Bottom Si Objective,0.766208,0.872084,0.910871,0.968748,1.290688


## 4.2. Keep runs with usable costs

Runs with missing or non-finite costs, along with runs at or above `COST_THRESHOLD`, are excluded from optimization.

In [39]:
print(
    f"Keeping runs with finite costs below {COST_THRESHOLD}. "
    "Missing and non-finite costs are also excluded.\n"
)

old_len = len(DF_opt)

usable_cost = (
    np.isfinite(DF_opt["Cost"])
    & (DF_opt["Cost"] < COST_THRESHOLD)
)
DF_opt = DF_opt.loc[usable_cost].reset_index(drop=True)

new_len = len(DF_opt)

print("Number of runs:", len(DF_opt))
print("Number of runs dropped:", old_len - new_len)

del old_len, new_len

Keeping runs with finite costs below 200. Missing and non-finite costs are also excluded.

Number of runs: 76
Number of runs dropped: 34


## 4.3. Identify varied numeric and categorical parameters

Numeric parameters are used by the continuous diagnostics in Section 9. The representative advection scheme is retained separately as a categorical parameter so it is never passed to numeric min/max, correlation, or regression calculations.

In [40]:
metadata_columns = {
    "Run Name", "Station File", "Parameter Status",
    "Cost File", "Cost", "Cost Status", RUN_DATE_COLUMN,
    *active_rmse_cols, *active_objective_cols,
}

candidate_parameter_cols = [
    column for column in DF_opt.columns
    if column not in metadata_columns
]

numeric_parameter_cols = [
    column for column in candidate_parameter_cols
    if pd.api.types.is_numeric_dtype(DF_opt[column])
]
categorical_parameter_cols = [
    column for column in candidate_parameter_cols
    if column not in numeric_parameter_cols
]

In [41]:
numeric_parameter_variation = (
    DF_opt[numeric_parameter_cols].nunique(dropna=True).sort_values()
)
categorical_parameter_variation = (
    DF_opt[categorical_parameter_cols].nunique(dropna=True).sort_values()
)

varied_parameter_cols = numeric_parameter_variation[
    numeric_parameter_variation > 1
].index.tolist()
varied_categorical_parameter_cols = categorical_parameter_variation[
    categorical_parameter_variation > 1
].index.tolist()
optimization_parameter_cols = (
    varied_parameter_cols + varied_categorical_parameter_cols
)

print("Varied numeric parameters:", len(varied_parameter_cols))
print(varied_parameter_cols)
print("\nVaried categorical parameters:", len(varied_categorical_parameter_cols))
print(varied_categorical_parameter_cols)

Varied numeric parameters: 40
['rrg1', 'rrg2', 'rrb1', 'wsp', 'pis2', 'aksio4s2', 'bgamma0', 'rrb2', 'bao2', 'bpsi_p', 'nl_tnu2_tracer9', 'bfs_nspc1', 'bfs_nspc0', 'wsdsi', 'amaxs1', 'Chl2cs1_m', 'aknh4s1', 'akno3s1', 'bgamma3', 'bgamma5s', 'gmaxs2', 'amaxs2', 'Chl2cs2_m', 'bgamma4', 'gmaxs1', 'akno3s2', 'reg1', 'reg2', 'akz2', 'akz1', 'bgamma6', 'beta1', 'beta2', 'bUmax_nspc0', 'bnit', 'wsd', 'bUmax_nspc1', 'bgamma7', 'bdenit', 'bgamma5']

Varied categorical parameters: 1
['tracer9_advection_scheme']


## 4.4. Build the Optuna dataframe

The optimization table retains the run name, total cost, parameters that vary, and the absolute values of the signed RMSD*' metrics. Total cost is the scalar Optuna objective; individual metrics are stored as trial metadata for diagnosis.

In [42]:
DF_optuna = DF_opt[
    ["Run Name", RUN_DATE_COLUMN, "Cost", *optimization_parameter_cols, *active_objective_cols]
].copy()

print(DF_optuna.shape)
display(DF_optuna.head())

(76, 56)


,Run Name,Run Date,Cost,rrg1,rrg2,rrb1,wsp,pis2,aksio4s2,bgamma0,...,Surface Oxygen Objective,Bottom Oxygen Objective,Surface NO3 Objective,Bottom NO3 Objective,Surface NH4 Objective,Bottom NH4 Objective,Surface Si Objective,Bottom Si Objective,Secchi Depth Objective,SOD Objective
0,DU_Mar16,2026-03-12,179.642600,0.1,0.1,0.2,0.4,1.59,2.0,0.025,...,0.733421,0.676851,3.266269,2.893021,2.761413,2.762684,0.991090,0.766208,0.751055,1.062886
1,DU,2026-03-07,59.782389,0.1,0.1,0.2,0.4,1.59,2.0,0.025,...,0.770595,0.835046,1.135046,1.065471,0.800463,1.186002,0.991277,0.970083,1.058675,1.063521
2,A17,2026-01-22,64.830081,0.2,0.2,0.3,0.3,1.59,1.0,0.050,...,0.467863,0.510462,0.924558,1.090406,0.824347,0.957371,0.747530,0.846235,1.240319,1.071099
3,C11,2026-04-23,59.965083,0.1,0.1,0.2,0.4,1.59,2.0,0.025,...,0.616898,0.674095,0.807021,0.886873,0.899166,1.040742,0.983584,1.254989,1.129166,0.950199
4,A16,2026-01-22,64.716917,0.2,0.2,0.3,0.3,1.59,1.0,0.050,...,0.470670,0.518845,0.904776,1.067467,0.810154,0.947802,0.755649,0.860693,1.240828,1.074462


# 5. Load optuna study

## 5.1. Load the study

[Watanabe (2023)](https://doi.org/10.48550/arXiv.2304.11127)

In [43]:
# ============================================================
# CREATE OR LOAD STUDY
# ============================================================

sampler = optuna.samplers.TPESampler(
    n_startup_trials=10,
    multivariate=True,
    seed=42
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction="minimize",
    sampler=sampler,
    load_if_exists=True
)

print(f"\nLoaded study: {study.study_name}")
print(f"Number of trials currently stored: {len(study.trials)}")

/var/folders/9k/5r38tm8d21g6nm3w9rchrd6m0000gn/T/ipykernel_84615/2460108777.py:5: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-09-16 13:28:50,247] A new study created in RDB with name: model_calibration_bio



Loaded study: model_calibration_bio
Number of trials currently stored: 0


In [44]:
# ============================================================
# VERIFY STUDY OBJECTIVE VERSION
# ============================================================

stored_objective_version = study.user_attrs.get(
    "Objective Version"
)

if stored_objective_version is None:

    study.set_user_attr(
        "Objective Version",
        OBJECTIVE_VERSION
    )

    print(
        f"Assigned objective version: "
        f"{OBJECTIVE_VERSION}"
    )

elif stored_objective_version != OBJECTIVE_VERSION:

    raise ValueError(
        "Objective-version mismatch:\n"
        f"  Study:    {stored_objective_version}\n"
        f"  Notebook: {OBJECTIVE_VERSION}\n\n"
        "Do not combine results calculated with different "
        "objective definitions in the same Optuna study."
    )

else:

    print(
        f"Objective version confirmed: "
        f"{OBJECTIVE_VERSION}"
    )

Assigned objective version: cost_v3_bio_no_temp_salt


## 5.2. Parameter distributions

Numeric ranges default to observed minima and maxima, with the existing manual `bdenit` override. The representative tracer-9 advection pair uses an explicit categorical distribution. Every active parameter must have a valid distribution before the study is modified.

In [45]:
# ============================================================
# NUMERIC PARAMETER LOOKUP TABLE
# ============================================================

parameter_summary = pd.DataFrame({
    "Parameter": varied_parameter_cols,
    "Low": [DF_optuna[parameter].min() for parameter in varied_parameter_cols],
    "High": [DF_optuna[parameter].max() for parameter in varied_parameter_cols],
    "Unique Values": [
        DF_optuna[parameter].nunique(dropna=True)
        for parameter in varied_parameter_cols
    ],
})
parameter_summary = parameter_summary.loc[
    parameter_summary["Unique Values"] > 1
].reset_index(drop=True)

all_parameter_bounds = {
    row["Parameter"]: (float(row["Low"]), float(row["High"]))
    for _, row in parameter_summary.iterrows()
}

# Existing scientific bound override.
if "bdenit" not in DF_optuna.columns:
    raise ValueError("Cannot override bdenit bounds: parameter is unavailable.")
all_parameter_bounds["bdenit"] = (1.0, float(DF_optuna["bdenit"].max()))

# ============================================================
# PARAMETERS TO OPTIMIZE
# ============================================================

active_numeric_parameters = [
    "bgamma7", "bgamma5", "bUmax_nspc1", "wsd", "bnit", "bdenit",
    "beta1", "beta2", "bgamma6", "bgamma3", "bgamma4", "reg1",
    "reg2", "rrb1", "rrb2", "rrg1", "rrg2",
    BIO_NL_TNU2_PARAMETER,
]
active_categorical_parameters = [BIO_ADVECTION_PARAMETER]
active_parameters = active_numeric_parameters + active_categorical_parameters

missing_numeric = [
    parameter for parameter in active_numeric_parameters
    if parameter not in all_parameter_bounds
]
if missing_numeric:
    raise ValueError(
        "Active numeric parameters lack usable historical bounds: "
        f"{missing_numeric}"
    )

parameter_bounds = {
    parameter: all_parameter_bounds[parameter]
    for parameter in active_numeric_parameters
}
invalid_bounds = {
    parameter: bounds
    for parameter, bounds in parameter_bounds.items()
    if not np.isfinite(bounds).all() or bounds[0] >= bounds[1]
}
if invalid_bounds:
    raise ValueError(f"Invalid Optuna parameter bounds: {invalid_bounds}")

categorical_choices = {
    BIO_ADVECTION_PARAMETER: tuple(BIO_ADVECTION_SCHEMES)
}

print("Numeric parameters being optimized:")
for parameter, (low, high) in parameter_bounds.items():
    print(f"  {parameter}: {low} - {high}")

print("\nCategorical parameters being optimized:")
for parameter, choices in categorical_choices.items():
    print(f"  {parameter}: {choices}")

Numeric parameters being optimized:
  bgamma7: 0.25 - 0.4961446158917879
  bgamma5: 0.15 - 0.3997598854239505
  bUmax_nspc1: 0.00274 - 0.02
  wsd: 2.0 - 5.0
  bnit: 0.15 - 0.2987851871698918
  bdenit: 1.0 - 28.979636342517402
  beta1: 0.5 - 2.25
  beta2: 0.15 - 1.75
  bgamma6: 0.005 - 0.1
  bgamma3: 0.1 - 0.25
  bgamma4: 0.05 - 0.3
  reg1: 0.1 - 0.2
  reg2: 0.05 - 0.1
  rrb1: 0.2 - 0.3
  rrb2: 0.2 - 0.3
  rrg1: 0.1 - 0.2
  rrg2: 0.1 - 0.2
  nl_tnu2_tracer9: 0.5 - 1.0

Categorical parameters being optimized:
  tracer9_advection_scheme: ('Upstream3_Centered4', 'HSIMT', 'Akima4', 'MPDATA')


## 5.3. Add historical runs

| Old Run Name | New Run Name | Station File |
| :--- | :------: | ----: |
| 2005_run1z_zi_modphys2_chlor2 | Dave1 | ocean_sta_Dave1_2005.nc |
| 2005_run1z_zi_modphys2_chlor2_chl2c | Dave2 | ocean_sta_Dave2_2005.nc |
| 2005_run1z_zj | Dave3 | ocean_sta_Dave3_2005.nc |
| 2005_run1z_zk | Dave4 | ocean_sta_Dave4_2005.nc |
| 2005_run1z_zl | Dave5 | ocean_sta_Dave5_2005.nc |
| 2005_run1z_zl_impl2 | Dave6 | ocean_sta_Dave6_2005.nc |

In [46]:
# ============================================================
# ADD HISTORICAL COMPLETED RUNS TO OPTUNA
# ============================================================

existing_run_names = {
    trial.user_attrs.get("Run Name")
    for trial in study.trials
    if trial.user_attrs.get("Run Name") is not None
}

float_distributions = {
    parameter: optuna.distributions.FloatDistribution(low=low, high=high)
    for parameter, (low, high) in parameter_bounds.items()
}
category_distributions = {
    parameter: optuna.distributions.CategoricalDistribution(choices=choices)
    for parameter, choices in categorical_choices.items()
}
trial_distributions = {**float_distributions, **category_distributions}

counts = {
    "Added": 0,
    "Already present": 0,
    "Missing parameters": 0,
    "Missing Cost": 0,
    "Outside bounds": 0,
    "Unknown category": 0,
}

# Stable sorting makes fresh-study trial numbers follow model run dates.
# Runs assigned the same date retain their order from DF_optuna.
historical_trials = DF_optuna.sort_values(
    RUN_DATE_COLUMN,
    kind="stable",
)

for _, row in historical_trials.iterrows():
    run_name = row["Run Name"]

    if run_name in existing_run_names:
        counts["Already present"] += 1
        continue
    if row[active_parameters].isna().any():
        counts["Missing parameters"] += 1
        continue
    if pd.isna(row["Cost"]) or not np.isfinite(row["Cost"]):
        counts["Missing Cost"] += 1
        continue
    if any(
        not (low <= float(row[parameter]) <= high)
        for parameter, (low, high) in parameter_bounds.items()
    ):
        counts["Outside bounds"] += 1
        continue
    if any(
        row[parameter] not in choices
        for parameter, choices in categorical_choices.items()
    ):
        counts["Unknown category"] += 1
        continue

    params = {
        parameter: float(row[parameter])
        for parameter in active_numeric_parameters
    }
    params.update({
        parameter: str(row[parameter])
        for parameter in active_categorical_parameters
    })

    objective_values = {
        objective: (
            float(row[objective])
            if pd.notna(row[objective]) and np.isfinite(row[objective])
            else None
        )
        for objective in active_objective_cols
    }

    completed_trial = optuna.trial.create_trial(
        params=params,
        distributions=trial_distributions,
        value=float(row["Cost"]),
        state=optuna.trial.TrialState.COMPLETE,
        user_attrs={
            "Run Name": run_name,
            "Source": "Historical",
            "Objective Version": OBJECTIVE_VERSION,
            "Objectives": objective_values,
            RUN_DATE_COLUMN: row[RUN_DATE_COLUMN].date().isoformat(),
        },
    )
    study.add_trial(completed_trial)
    existing_run_names.add(run_name)
    counts["Added"] += 1

print("Historical trial import:")
for label, count in counts.items():
    print(f"  {label}: {count}")
print(f"\nTotal Optuna trials: {len(study.trials)}")

Historical trial import:
  Added: 75
  Already present: 1
  Missing parameters: 0
  Missing Cost: 0
  Outside bounds: 0
  Unknown category: 0

Total Optuna trials: 75


# 6. Candidates

## 6.1. Register newly completed model runs

A stored `RUNNING` trial is eligible for completion when its mapped actual run name appears in the cleaned results table with a finite cost. If the trial is absent from the current study's run map, its existing run name is used. Mapped study run names and trial numbers must agree. Eligible trials are always listed. The Optuna database is modified only when `UPDATE_OPTUNA_WITH_COMPLETED_RUNS` is `True`. This synchronization occurs before new candidates are requested so completed trials do not unnecessarily block the next batch.

In [47]:
# ============================================================
# FIND AND OPTIONALLY COMPLETE FINISHED RUNNING TRIALS
# ============================================================

duplicate_mask = DF_optuna["Run Name"].duplicated(keep=False)
duplicate_names = DF_optuna.loc[duplicate_mask, "Run Name"].unique()

conflicting_duplicate_names = []
for run_name in duplicate_names:
    duplicate_rows = DF_optuna.loc[
        DF_optuna["Run Name"] == run_name
    ]
    if len(duplicate_rows.drop_duplicates()) > 1:
        conflicting_duplicate_names.append(run_name)

if conflicting_duplicate_names:
    raise ValueError(
        "Duplicate run names have conflicting optimization values: "
        f"{conflicting_duplicate_names}"
    )

if len(duplicate_names) > 0:
    print(
        "Removing exact duplicate optimization rows for: "
        f"{duplicate_names.tolist()}"
    )
    DF_optuna = DF_optuna.drop_duplicates(
        subset=["Run Name"],
        keep="first",
    ).reset_index(drop=True)

results_by_run = DF_optuna.set_index("Run Name", drop=False)
run_map_by_study_run_name = current_study_run_map.set_index(
    "Study Run Name", drop=False
)
run_map_by_trial_number = current_study_run_map.set_index(
    "Trial Number", drop=False
)
running_before_update = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]

completion_rows = []
n_completed_now = 0
n_results_not_available = 0
n_missing_run_name = 0

for frozen_trial in running_before_update:
    study_run_name = frozen_trial.user_attrs.get("Run Name")

    if not study_run_name:
        n_missing_run_name += 1
        print(f"Trial {frozen_trial.number}: missing Run Name attribute")
        continue

    if study_run_name in run_map_by_study_run_name.index:
        mapping = run_map_by_study_run_name.loc[study_run_name]
        mapped_trial_number = int(mapping["Trial Number"])
        if mapped_trial_number != frozen_trial.number:
            raise ValueError(
                f"Run-map mismatch for study {CURRENT_STUDY_NAME!r}: "
                f"Study Run Name {study_run_name!r} maps to trial "
                f"{mapped_trial_number}, but the RUNNING Optuna trial is "
                f"{frozen_trial.number}."
            )
        actual_run_name = mapping["Actual Run Name"]
    elif frozen_trial.number in run_map_by_trial_number.index:
        mapping = run_map_by_trial_number.loc[frozen_trial.number]
        raise ValueError(
            f"Run-map mismatch for study {CURRENT_STUDY_NAME!r}: trial "
            f"{frozen_trial.number} maps to Study Run Name "
            f"{mapping['Study Run Name']!r}, but the trial stores "
            f"{study_run_name!r}."
        )
    else:
        # The run map is an override table; unmapped trials keep existing behavior.
        actual_run_name = study_run_name

    if actual_run_name not in results_by_run.index:
        n_results_not_available += 1
        continue

    row = results_by_run.loc[actual_run_name]
    cost = float(row["Cost"])
    objective_values = {
        objective: (
            float(row[objective])
            if pd.notna(row[objective]) and np.isfinite(row[objective])
            else None
        )
        for objective in active_objective_cols
    }

    completion_rows.append({
        "Study Name": CURRENT_STUDY_NAME,
        "Actual Run Name": actual_run_name,
        "Study Run Name": study_run_name,
        "Trial Number": frozen_trial.number,
        RUN_DATE_COLUMN: row[RUN_DATE_COLUMN],
        "Cost": cost,
        "Action": (
            "Complete in Optuna"
            if UPDATE_OPTUNA_WITH_COMPLETED_RUNS
            else "Preview only"
        ),
    })

    if not UPDATE_OPTUNA_WITH_COMPLETED_RUNS:
        continue

    # Recreate a live Trial solely to attach metadata before study.tell().
    live_trial = optuna.trial.Trial(study, frozen_trial._trial_id)
    live_trial.set_user_attr("Study Run Name", study_run_name)
    live_trial.set_user_attr("Actual Run Name", actual_run_name)
    live_trial.set_user_attr("Objectives", objective_values)
    live_trial.set_user_attr(
        RUN_DATE_COLUMN, row[RUN_DATE_COLUMN].date().isoformat()
    )
    study.tell(frozen_trial.number, cost)
    n_completed_now += 1

DF_completion_candidates = pd.DataFrame(
    completion_rows,
    columns=[
        "Study Name", "Actual Run Name", "Study Run Name",
        "Trial Number", RUN_DATE_COLUMN, "Cost", "Action",
    ],
)

if DF_completion_candidates.empty:
    print("No RUNNING trials have completed results available.")
else:
    display(DF_completion_candidates)

print("\nOptuna completion summary:")
print(f"  Update enabled:          {UPDATE_OPTUNA_WITH_COMPLETED_RUNS}")
print(f"  Eligible for completion: {len(completion_rows)}")
print(f"  Completed now:           {n_completed_now}")
print(f"  Results not available:   {n_results_not_available}")
print(f"  Missing Run Name:        {n_missing_run_name}")

Removing exact duplicate optimization rows for: ['LHS20_2005']
No RUNNING trials have completed results available.

Optuna completion summary:
  Update enabled:          False
  Eligible for completion: 0
  Completed now:           0
  Results not available:   0
  Missing Run Name:        0


In [48]:
# ============================================================
# REPORT STUDY STATUS AND LOAD CURRENT RUNNING CANDIDATES
# ============================================================

print(f"Study: {study.study_name}")
print(f"Total trials: {len(study.trials)}")

for state in optuna.trial.TrialState:
    count = sum(trial.state == state for trial in study.trials)
    if count > 0:
        print(f"  {state.name}: {count}")

running_trials = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]

print(f"Running trials: {len(running_trials)}")
candidate_results = []

for trial in running_trials:
    
    print(
        f"  Trial {trial.number}: "
        f"{trial.user_attrs.get('Run Name', 'No Run Name')}"
    )

    candidate_results.append({
        "Run Name": trial.user_attrs.get("Run Name"),
        "Trial Number": trial.number,
        **trial.params
    })

DF_candidates = pd.DataFrame(
    candidate_results,
    columns=["Run Name", "Trial Number", *active_parameters],
)

display(DF_candidates)

Study: model_calibration_bio
Total trials: 75
  COMPLETE: 75
Running trials: 0


,Run Name,Trial Number,bgamma7,bgamma5,bUmax_nspc1,wsd,bnit,bdenit,beta1,beta2,...,bgamma3,bgamma4,reg1,reg2,rrb1,rrb2,rrg1,rrg2,nl_tnu2_tracer9,tracer9_advection_scheme


## 6.2. Generate candidate parameter sets

New trials are requested only when `GENERATE_NEW_CANDIDATES` is `True` and no trials remain `RUNNING`. The setting itself is never changed by the notebook. Existing running candidates remain available for input-file regeneration.

In [ ]:
# ============================================================
# GENERATE NEW OPTUNA CANDIDATES
# ============================================================

running_trials = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]
generated_new_candidates = False

if GENERATE_NEW_CANDIDATES and running_trials:
    print(
        f"Not generating candidates because {len(running_trials)} "
        "trial(s) are already RUNNING."
    )

elif GENERATE_NEW_CANDIDATES:
    candidate_results = []

    for _ in range(N_CANDIDATES):
        trial = study.ask()
        run_name = f"OPTUNA_BIO_{trial.number}"
        trial.set_user_attr("Run Name", run_name)
        trial.set_user_attr("Source", "Optuna suggestion")
        trial.set_user_attr("Objective Version", OBJECTIVE_VERSION)

        params = {
            parameter: trial.suggest_float(parameter, low, high)
            for parameter, (low, high) in parameter_bounds.items()
        }
        for parameter, choices in categorical_choices.items():
            params[parameter] = trial.suggest_categorical(parameter, choices)

        candidate_results.append({
            "Run Name": run_name,
            "Trial Number": trial.number,
            **params,
        })

    DF_candidates = pd.DataFrame(candidate_results)
    generated_new_candidates = True
    display(DF_candidates)

else:
    print("Candidate generation is OFF.")
    if running_trials:
        print(f"{len(running_trials)} trial(s) remain RUNNING.")

In [ ]:
# ============================================================
# SAVE CANDIDATE PARAMETER SETS
# ============================================================

if generated_new_candidates:

    candidate_file = OPT_DIR / "Optuna_Bio_Candidate_Parameters.csv"

    DF_candidates.to_csv(
        candidate_file,
        index=False
    )

    print("Saved candidate parameters to:")
    print(candidate_file)

# 7. Create parameter input files

## 7.1. Files and templates

Templates are read for parameter auditing even when file writing is disabled. `OUTPUT_DIR` is created only when `WRITE_CANDIDATE_INPUT_FILES` is `True` and at least one running candidate is available.

In [ ]:
# ============================================================
# TEMPLATE INPUT FILES
# ============================================================

OPTICS_TEMPLATE = Path(
    "/Users/akbaskind/Desktop/LHS/"
    "bio_UMAINE15_sediment_optics_LHS1.in"
)

SEDIMENT_TEMPLATE = Path(
    "/Users/akbaskind/Desktop/LHS/"
    "bio_UMAINE15_sedmodel_LHS1.in"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = OPT_DIR / "Candidate_Input_Files"


def format_roms_value(value):
    """Format a numeric ROMS input value with D exponent notation."""

    text = f"{float(value):.16g}"

    if "." not in text and "e" not in text.lower():
        text += ".0"

    return f"{text}d0"


def replace_biological_tnu2(text, value):
    """Apply one nl_tnu2 value to all biological tracers."""

    if not np.isfinite(value):
        raise ValueError(f"nl_tnu2 has non-finite value {value}")

    pattern = (
        rf"^(\s*TNU2\s*==\s*{BIOLOGICAL_TRACER_COUNT}\s*\*\s*)"
        rf"([^\s!]+)"
    )
    updated, count = re.subn(
        pattern,
        rf"\g<1>{format_roms_value(value)}",
        text,
        count=1,
        flags=re.MULTILINE,
    )
    if count != 1:
        raise ValueError(f"TNU2 had {count} active-line replacements")
    return updated


def replace_advection_block(text, keyword, scheme_code, expected_values=15):
    """Replace every value in one Hadvection or Vadvection block."""

    lines = text.splitlines(keepends=True)
    start_indices = [
        index for index, line in enumerate(lines)
        if re.match(rf"^\s*{re.escape(keyword)}\s*==", line)
    ]
    if len(start_indices) != 1:
        raise ValueError(f"{keyword} had {len(start_indices)} active blocks")

    start = start_indices[0]
    for offset in range(expected_values):
        index = start + offset
        if index >= len(lines):
            raise ValueError(f"{keyword} ended before value {offset + 1}")

        if offset == 0:
            pattern = rf"^(\s*{re.escape(keyword)}\s*==\s*)(\S+)(.*)$"
        else:
            pattern = r"^(\s*)(\S+)(\s+.*)$"

        updated, count = re.subn(pattern, rf"\g<1>{scheme_code}\g<3>", lines[index], count=1)
        if count != 1 or f"idbio({offset + 1:2d})" not in updated:
            raise ValueError(
                f"{keyword} value {offset + 1} did not match the expected template line"
            )
        lines[index] = updated

    return "".join(lines)


## 7.2. Lookup table for ALL parameters

In [ ]:
# ============================================================
# PARAMETERS IN EACH INPUT FILE
# ============================================================

optics_parameters = [
    "reg1",
    "reg2",
    "gmaxs1",
    "gmaxs2",
    "rrb1",
    "rrb2",
    "rrg1",
    "rrg2",
    "beta1",
    "beta2",
    "akz1",
    "akz2",
    "PARfrac",
    "amaxs1",
    "amaxs2",
    "parsats1",
    "parsats2",
    "pis1",
    "pis2",
    "akno3s1",
    "akno3s2",
    "aknh4s1",
    "aknh4s2",
    "akpo4s1",
    "akpo4s2",
    "akco2s1",
    "akco2s2",
    "aksio4s2",
    "ak1",
    "ak2",
    "bgamma0",
    "bgamma1",
    "bgamma2",
    "bgamma3",
    "bgamma4",
    "bgamma5",
    "bgamma5s",
    "bgamma6",
    "bgamma7",
    "wsd",
    "wsdsi",
    "wsp",
    "pco2a",
    "si2n",
    "p2n",
    "o2no",
    "o2nh",
    "c2n",
    "ro5",
    "ro6",
    "ro7",
    "akox",
    "Chl2cs1_m",
    "Chl2cs2_m",
]

sediment_parameters = [
     'bUmax_nspc0',
     'bUmax_nspc1',
     'bUmax_nspc2',
     'bUmaxSi_nspc0',
     'bUmaxSi_nspc1',
     'bUmaxSi_nspc2',
     'bdep',
     'balpha',
     'bw',
     'btheta_diag',
     'bnit',
     'btheta_nit',
     'bdo_nit',
     'bdenit',
     'btheta_denit',
     'bpsi_n',
     'bdo_c',
     'bao2',
     'bpi',
     'bpsi_p',
     'bfc_nspc0',
     'bfc_nspc1',
     'bfc_nspc2',
     'bfn_nspc0',
     'bfn_nspc1',
     'bfn_nspc2',
     'bfp_nspc0',
     'bfp_nspc1',
     'bfp_nspc2',
     'bfs_nspc0',
     'bfs_nspc1',
     'bfs_nspc2'
]

## 7.3. Active parameters

These are the active Optuna parameters assigned to each template. The preparation flag is separate from candidate generation, so files can be recreated for trials that are already `RUNNING`.

`nl_tnu2_tracer9` and `tracer9_advection_scheme` are special water-column controls. They are handled by validated block replacements rather than the scalar biological-parameter replacement loop.


In [ ]:
active_optics_parameters = [
    p for p in active_parameters
    if p in optics_parameters
]

active_sediment_parameters = [
    p for p in active_parameters
    if p in sediment_parameters
]

special_optics_parameters = [
    BIO_NL_TNU2_PARAMETER,
    BIO_ADVECTION_PARAMETER,
]

print("Active optics parameters:")
print(active_optics_parameters)

print("\nActive sediment parameters:")
print(active_sediment_parameters)

prepare_candidate_input_files = (
    WRITE_CANDIDATE_INPUT_FILES
    and not DF_candidates.empty
)

if WRITE_CANDIDATE_INPUT_FILES and DF_candidates.empty:
    print("No candidate input files will be written: no RUNNING candidates.")

print("\nSpecial optics parameters:")
print(special_optics_parameters)


## 7.4. Setting inactive parameters from baseline run with scientific overrides

The baseline run is the best performing historical run. The parameters Optuna is NOT varying will be set to the value used in the baseline run, barring certain overrides, which were specified earlier.

In [ ]:
# ============================================================
# BUILD FIXED CANDIDATE BASELINE
# ============================================================

baseline_matches = DF_opt.loc[
    DF_opt["Run Name"] == BASELINE_RUN_NAME
]

if len(baseline_matches) == 0:
    raise ValueError(
        f"Baseline run {BASELINE_RUN_NAME!r} was not found."
    )

if len(baseline_matches) > 1:
    raise ValueError(
        f"Multiple rows were found for {BASELINE_RUN_NAME!r}."
    )

baseline_run = baseline_matches.iloc[0]

# All parameters represented in the two candidate input files
input_file_parameters = (
    optics_parameters
    + sediment_parameters
)

# Parameters that baseline can supply
baseline_values = {
    parameter: float(baseline_run[parameter])
    for parameter in input_file_parameters
    if (
        parameter in DF_opt.columns
        and pd.notna(baseline_run[parameter])
    )
}

# Scientific overrides must remain fixed, not varied by Optuna
override_conflicts = (
    set(SCIENTIFIC_OVERRIDES)
    & set(active_parameters)
)

if override_conflicts:
    raise ValueError(
        "These parameters are both scientific overrides "
        "and active Optuna parameters: "
        f"{sorted(override_conflicts)}"
    )

# Start with Dave5, but exclude parameters Optuna will vary
fixed_candidate_values = {
    parameter: value
    for parameter, value in baseline_values.items()
    if parameter not in active_parameters
}

# Scientific decisions take precedence over Dave5
fixed_candidate_values.update(
    SCIENTIFIC_OVERRIDES
)

print(f"Baseline run: {BASELINE_RUN_NAME}")
print(f"Dave5 parameter values found: {len(baseline_values)}")
print(f"Fixed candidate values:       {len(fixed_candidate_values)}")

print("\nScientific overrides:")
for parameter, value in SCIENTIFIC_OVERRIDES.items():
    print(f"  {parameter}: {value}")

### 7.4.1. Audit parameter sources

In [ ]:
# ============================================================
# AUDIT CANDIDATE PARAMETER SOURCES
# ============================================================

unknown_overrides = (
    set(SCIENTIFIC_OVERRIDES)
    - set(input_file_parameters)
)

if unknown_overrides:
    raise ValueError(
        "Scientific overrides not found in either input file: "
        f"{sorted(unknown_overrides)}"
    )

audit_rows = []

# dict.fromkeys removes duplicates while preserving order
for parameter in dict.fromkeys(input_file_parameters):

    if parameter in optics_parameters:
        input_file = "Optics"
    else:
        input_file = "Sediment"

    baseline_value = baseline_values.get(
        parameter,
        np.nan
    )

    override_value = SCIENTIFIC_OVERRIDES.get(
        parameter,
        np.nan
    )

    if parameter in active_parameters:

        source = "Optuna"
        candidate_value = "Varies by Optuna"

    elif parameter in SCIENTIFIC_OVERRIDES:

        source = "Scientific override"
        candidate_value = SCIENTIFIC_OVERRIDES[parameter]

    elif parameter in baseline_values:

        source = BASELINE_RUN_NAME
        candidate_value = baseline_values[parameter]

    else:

        source = "Current template"
        candidate_value = "Unchanged"

    audit_rows.append({
        "Parameter": parameter,
        "Input File": input_file,
        "Baseline Value": baseline_value,
        "Scientific Override": override_value,
        "Candidate Source": source,
        "Candidate Value": candidate_value
    })

for parameter in special_optics_parameters:
    audit_rows.append({
        "Parameter": parameter,
        "Input File": "Optics (special block)",
        "Baseline Value": baseline_run.get(parameter, np.nan),
        "Scientific Override": np.nan,
        "Candidate Source": "Optuna",
        "Candidate Value": "Varies by Optuna",
    })

DF_parameter_audit = pd.DataFrame(audit_rows)

display(DF_parameter_audit)

### 7.4.2. Read default parameter values from template

In [ ]:
# ============================================================
# READ A PARAMETER VALUE FROM AN INPUT-FILE TEMPLATE
# ============================================================

def read_template_parameter(text, parameter):

    # Handle array values such as bUmax_nspc1
    if "_nspc" in parameter:
        base_parameter, index_text = parameter.split("_nspc")
        value_index = int(index_text)
    else:
        base_parameter = parameter
        value_index = 0

    pattern = (
        rf"^\s*{re.escape(base_parameter)}"
        rf"\s*==\s*([^!\n]+)"
    )

    match = re.search(
        pattern,
        text,
        flags=re.MULTILINE
    )

    if match is None:
        return np.nan

    values_text = match.group(1)

    number_pattern = (
        r"[-+]?"
        r"(?:\d+(?:\.\d*)?|\.\d+)"
        r"(?:[dDeE][-+]?\d+)?"
    )

    number_strings = re.findall(
        number_pattern,
        values_text
    )

    if value_index >= len(number_strings):
        return np.nan

    value_text = (
        number_strings[value_index]
        .replace("D", "e")
        .replace("d", "e")
    )

    return float(value_text)

In [ ]:
# ============================================================
# EXTRACT VALUES FROM CURRENT TEMPLATES
# ============================================================

current_optics_text = OPTICS_TEMPLATE.read_text()
current_sediment_text = SEDIMENT_TEMPLATE.read_text()

current_template_values = {}

for parameter in optics_parameters:

    current_template_values[parameter] = (
        read_template_parameter(
            current_optics_text,
            parameter
        )
    )

for parameter in sediment_parameters:

    current_template_values[parameter] = (
        read_template_parameter(
            current_sediment_text,
            parameter
        )
    )

print(
    "Current template values found:",
    sum(
        pd.notna(value)
        for value in current_template_values.values()
    )
)

### 7.4.3. Parameters differing from baseline and template

In [ ]:
# ============================================================
# PARAMETERS THAT DIFFER BETWEEN CURRENT TEMPLATE AND DAVE5
# ============================================================

different_parameter_rows = []

for parameter in input_file_parameters:

    current_value = current_template_values.get(
        parameter,
        np.nan
    )

    baseline_value = baseline_values.get(
        parameter,
        np.nan
    )

    # A comparison requires both values
    if pd.isna(current_value) or pd.isna(baseline_value):
        continue

    if not np.isclose(
        current_value,
        baseline_value,
        rtol=1e-9,
        atol=1e-12
    ):

        difference = baseline_value - current_value

        if current_value != 0:
            percent_difference = (
                difference / abs(current_value)
            ) * 100
        else:
            percent_difference = np.nan

        if parameter in SCIENTIFIC_OVERRIDES:
            planned_source = "Scientific override"
            planned_value = SCIENTIFIC_OVERRIDES[parameter]

        elif parameter in active_parameters:
            planned_source = "Optuna"
            planned_value = "Varies by Optuna"

        else:
            planned_source = BASELINE_RUN_NAME
            planned_value = baseline_value

        different_parameter_rows.append({
            "Parameter": parameter,
            "Current Template": current_value,
            "Baseline": baseline_value,
            "Difference": difference,
            "Percent Difference": percent_difference,
            "Planned Source": planned_source,
            "Planned Candidate Value": planned_value
        })

DF_different_parameters = pd.DataFrame(
    different_parameter_rows
)

if not DF_different_parameters.empty:

    DF_different_parameters = (
        DF_different_parameters
        .sort_values(
            "Percent Difference",
            key=lambda values: values.abs(),
            ascending=False,
            na_position="last"
        )
        .reset_index(drop=True)
    )

print(
    "Parameters that differ between the current "
    f"templates and {BASELINE_RUN_NAME}: "
    f"{len(DF_different_parameters)}"
)

display(DF_different_parameters)

### 7.4.4. Fixed parameters

In [ ]:
fixed_optics_parameters = [
    parameter
    for parameter in fixed_candidate_values
    if parameter in optics_parameters
]

fixed_sediment_parameters = [
    parameter
    for parameter in fixed_candidate_values
    if parameter in sediment_parameters
]

print("Fixed optics parameters:")
print(fixed_optics_parameters)

print("\nFixed sediment parameters:")
print(fixed_sediment_parameters)

### 7.4.5. Varying parameters

In [ ]:
if DF_candidates.empty:
    print("No RUNNING candidates to preview.")
else:
    candidate = DF_candidates.iloc[0]

    print(f"Run Name: {candidate['Run Name']}")
    print("\nOptics parameters:")
    for parameter in active_optics_parameters:
        print(f"{parameter}: {candidate[parameter]}")

    print("\nSediment parameters:")
    for parameter in active_sediment_parameters:
        print(f"{parameter}: {candidate[parameter]}")

    print("\nRepresentative biological-tracer settings:")
    for parameter in special_optics_parameters:
        print(f"{parameter}: {candidate[parameter]}")


## 7.5. Generate water-column input text

In [ ]:
# ============================================================
# GENERATE OPTICS FILE TEXT FOR ALL OPTUNA CANDIDATES
# ============================================================

if prepare_candidate_input_files:

    all_optics_text = {}

    for _, candidate in DF_candidates.iterrows():

        run_name = candidate["Run Name"]

        # Start fresh from the original template
        optics_text = OPTICS_TEMPLATE.read_text()

        # ----------------------------------------------------
        # Apply fixed baseline and scientific overrides
        # ----------------------------------------------------

        for parameter in fixed_optics_parameters:

            new_value = fixed_candidate_values[parameter]

            pattern = (
                rf"^(\s*{re.escape(parameter)}"
                rf"\s*==\s*)([^\s!]+)"
            )

            optics_text, n = re.subn(
                pattern,
                rf"\g<1>{format_roms_value(new_value)}",
                optics_text,
                count=1,
                flags=re.MULTILINE
            )

            if n != 1:
                raise ValueError(
                    f"{run_name}: {parameter} had "
                    f"{n} fixed-value replacements"
                )

        # ----------------------------------------------------
        # Apply Optuna-suggested values
        # ----------------------------------------------------

        for parameter in active_optics_parameters:

            new_value = float(candidate[parameter])

            pattern = (
                rf"^(\s*{re.escape(parameter)}"
                rf"\s*==\s*)([^\s!]+)"
            )

            optics_text, n = re.subn(
                pattern,
                rf"\g<1>{format_roms_value(new_value)}",
                optics_text,
                count=1,
                flags=re.MULTILINE
            )

            if n != 1:
                raise ValueError(
                    f"{run_name}: {parameter} had "
                    f"{n} Optuna-value replacements"
                )

        # ----------------------------------------------------
        # Apply representative biological-tracer settings
        # uniformly to all 15 biological tracers
        # ----------------------------------------------------

        optics_text = replace_biological_tnu2(
            optics_text,
            float(candidate[BIO_NL_TNU2_PARAMETER]),
        )

        advection_category = candidate[BIO_ADVECTION_PARAMETER]
        if advection_category not in BIO_ADVECTION_SCHEMES:
            raise ValueError(
                f"{run_name}: unknown advection category {advection_category!r}"
            )

        horizontal_name, vertical_name = BIO_ADVECTION_SCHEMES[advection_category]
        horizontal_code = ADVECTION_INPUT_CODES[horizontal_name]
        vertical_code = ADVECTION_INPUT_CODES[vertical_name]

        optics_text = replace_advection_block(
            optics_text, "Hadvection", horizontal_code, BIOLOGICAL_TRACER_COUNT
        )
        optics_text = replace_advection_block(
            optics_text, "Vadvection", vertical_code, BIOLOGICAL_TRACER_COUNT
        )

        # ----------------------------------------------------
        # Update BSEDPARNAM
        # ----------------------------------------------------

        sediment_filename = (
            f"bio_UMAINE15_sedmodel_{run_name}.in"
        )

        sediment_path = (
            "/home/abaskind_uri_edu/"
            "ROMS_forcing_files/input_files/OPTUNA/"
            f"{sediment_filename}"
        )

        pattern = r"^(\s*BSEDPARNAM\s*==\s*)(\S+)"

        optics_text, n = re.subn(
            pattern,
            rf"\g<1>{sediment_path}",
            optics_text,
            count=1,
            flags=re.MULTILINE
        )

        if n != 1:
            raise ValueError(
                f"{run_name}: BSEDPARNAM had "
                f"{n} replacements"
            )

        all_optics_text[run_name] = optics_text

        print(f"Finished {run_name}")

### 7.5.1. Inspect generated water-column text

In [ ]:
if prepare_candidate_input_files:
    run_name = DF_candidates.iloc[0]["Run Name"]
    lines = all_optics_text[run_name].splitlines()

    for index, line in enumerate(lines):
        if any(line.strip().startswith(parameter) for parameter in active_optics_parameters):
            print(line)
        if line.strip().startswith("TNU2"):
            print(line)
        if line.strip().startswith("Hadvection"):
            print("\n".join(lines[index:index + BIOLOGICAL_TRACER_COUNT]))
        if line.strip().startswith("Vadvection"):
            print("\n".join(lines[index:index + BIOLOGICAL_TRACER_COUNT]))
        if line.strip().startswith("BSED"):
            print(line)

## 7.6. Generate sediment input text

### 7.6.1. Replacement function

In [ ]:
def replace_sediment_parameter(text, parameter, value):

    if not np.isfinite(value):
        raise ValueError(f"{parameter} has non-finite value {value}")

    # --------------------------------
    # Multi-valued parameter
    # Example: bUmax_nspc1
    # --------------------------------

    if "_nspc" in parameter:

        base_parameter, index_text = parameter.split("_nspc")

        index = int(index_text)

        pattern = rf"^(\s*{re.escape(base_parameter)}\s*==\s*)([^!\n]+)(.*)$"

        match = re.search(
            pattern,
            text,
            flags=re.MULTILINE
        )

        if match is None:
            raise ValueError(
                f"Could not find {base_parameter}"
            )

        prefix = match.group(1)
        values_text = match.group(2)
        suffix = match.group(3)

        values = values_text.split()

        if index >= len(values):
            raise ValueError(
                f"{parameter} requests index {index}, but "
                f"{base_parameter} contains {len(values)} value(s)"
            )

        values[index] = format_roms_value(value)

        replacement = (
            prefix
            + " ".join(values)
            + suffix
        )

        text = (
            text[:match.start()]
            + replacement
            + text[match.end():]
        )

    # --------------------------------
    # Scalar parameter
    # Example: bnit
    # --------------------------------

    else:

        pattern = rf"^(\s*{re.escape(parameter)}\s*==\s*)([^\s!]+)"

        text, n = re.subn(
            pattern,
            rf"\g<1>{format_roms_value(value)}",
            text,
            count=1,
            flags=re.MULTILINE
        )

        if n != 1:
            raise ValueError(
                f"{parameter} had {n} replacements"
            )

    return text

### 7.6.2. Generate candidate text

In [ ]:
# ============================================================
# GENERATE SEDIMENT FILE TEXT FOR ALL OPTUNA CANDIDATES
# ============================================================

if prepare_candidate_input_files:

    all_sediment_text = {}

    for _, candidate in DF_candidates.iterrows():

        run_name = candidate["Run Name"]

        # Start fresh from the original sediment template
        sediment_text = SEDIMENT_TEMPLATE.read_text()

        # ----------------------------------------------------
        # Apply fixed baseline and scientific overrides
        # ----------------------------------------------------

        for parameter in fixed_sediment_parameters:

            sediment_text = replace_sediment_parameter(
                sediment_text,
                parameter,
                fixed_candidate_values[parameter]
            )

        # ----------------------------------------------------
        # Apply Optuna-suggested values
        # ----------------------------------------------------

        for parameter in active_sediment_parameters:

            sediment_text = replace_sediment_parameter(
                sediment_text,
                parameter,
                float(candidate[parameter])
            )

        all_sediment_text[run_name] = sediment_text

        print(f"Finished {run_name}")

### 7.6.3. Inspect generated sediment text

In [ ]:
if prepare_candidate_input_files:
    run_name = DF_candidates.iloc[0]["Run Name"]

    for line in all_sediment_text[run_name].splitlines():

        if (
            line.strip().startswith("bUmax")
            or line.strip().startswith("bnit")
            or line.strip().startswith("bdenit")
        ):
            print(line)

## 7.7. Write candidate files

Writing is explicit and occurs only after both generated texts have passed their replacement-count checks. Existing files with the same candidate names are replaced. Numeric and advection block replacements must each pass exact-count validation first.

In [ ]:
# ============================================================
# WRITE MODEL INPUT FILES FOR ALL OPTUNA CANDIDATES
# ============================================================
if prepare_candidate_input_files:

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    generated_files = []

    for _, candidate in DF_candidates.iterrows():

        run_name = candidate["Run Name"]

        # --------------------------------------------------------
        # File names
        # --------------------------------------------------------

        optics_filename = (
            f"bio_UMAINE15_sediment_optics_{run_name}.in"
        )

        sediment_filename = (
            f"bio_UMAINE15_sedmodel_{run_name}.in"
        )

        optics_output = OUTPUT_DIR / optics_filename
        sediment_output = OUTPUT_DIR / sediment_filename

        # --------------------------------------------------------
        # Write the already-modified text
        # --------------------------------------------------------

        optics_output.write_text(
            all_optics_text[run_name]
        )

        sediment_output.write_text(
            all_sediment_text[run_name]
        )

        # --------------------------------------------------------
        # Keep a record
        # --------------------------------------------------------

        generated_files.append({
            "Run Name": run_name,
            "Optics File": optics_filename,
            "Sediment File": sediment_filename
        })

        print(f"Wrote {run_name}")

### 7.7.1. Verify written files

In [ ]:
if prepare_candidate_input_files:
    DF_generated_files = pd.DataFrame(generated_files)

    display(DF_generated_files)

In [ ]:
if prepare_candidate_input_files:
    print("Files written to:")
    print(OUTPUT_DIR)

    print("\nNumber of optics files:")
    print(len(list(OUTPUT_DIR.glob("bio_UMAINE15_sediment_optics_*.in"))))

    print("\nNumber of sediment files:")
    print(len(list(OUTPUT_DIR.glob("bio_UMAINE15_sedmodel_*.in"))))

# 8. Final study status and export

Review the final trial states after optional synchronization and candidate generation. The cleaned run summary is written only when `EXPORT_RUN_SUMMARY` is `True`.

In [49]:
# ============================================================
# FINAL STUDY STATUS
# ============================================================

print(f"Study: {study.study_name}")
print(f"Total trials: {len(study.trials)}")

for state in optuna.trial.TrialState:
    count = sum(trial.state == state for trial in study.trials)
    if count > 0:
        print(f"  {state.name}: {count}")

Study: model_calibration_bio
Total trials: 75
  COMPLETE: 75


In [50]:
running_trials = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]

print(f"\nTrials still RUNNING: {len(running_trials)}")

for trial in running_trials:
    print(
        f"  Trial {trial.number}: "
        f"{trial.user_attrs.get('Run Name')}"
    )


Trials still RUNNING: 0


In [ ]:
# ============================================================
# EXPORT VALIDATED RUN SUMMARY FOR PARAMETER ANALYSIS
# ============================================================

RUN_SUMMARY_FILE = OPT_DIR / "Model_Run_Summary_Bio.csv"

DF_run_summary = DF_opt.copy()

DF_run_summary["Objective Version"] = OBJECTIVE_VERSION

# Match each run to its actual Optuna trial number
trial_number_by_run = {
    trial.user_attrs.get("Run Name"): trial.number
    for trial in study.trials
    if trial.user_attrs.get("Run Name") is not None
}

DF_run_summary["Trial Number"] = (
    DF_run_summary["Run Name"]
    .map(trial_number_by_run)
    .astype("Int64")
)

if EXPORT_RUN_SUMMARY:
    DF_run_summary.to_csv(
        RUN_SUMMARY_FILE,
        index=False,
    )
    print("Saved validated model-run summary:")
    print(RUN_SUMMARY_FILE)
else:
    print("Run-summary export is OFF.")

print(f"\nRows:    {len(DF_run_summary)}")
print(f"Columns: {len(DF_run_summary.columns)}")
print(
    "Runs matched to Optuna trials:",
    DF_run_summary["Trial Number"].notna().sum()
)